In [7]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np

In [3]:

# URLs de los archivos
url_train = "https://practicum-content.s3.us-west-1.amazonaws.com/datasets/gold_recovery_train.csv?etag=68f7294d2022296464fd4d705519843c"
url_test = "https://practicum-content.s3.us-west-1.amazonaws.com/datasets/gold_recovery_test.csv?etag=1e251eb453e155475fca8d03d8b66ae2"
url_full = "https://practicum-content.s3.us-west-1.amazonaws.com/datasets/gold_recovery_full.csv?etag=b2fba00139bca2b8c4c9af43667e0656"

# Cargar los datasets

df_train = (
    pd.read_csv(url_train, parse_dates=['date'])
    .sort_values('date')
    .set_index('date')
)

df_test = (
    pd.read_csv(url_test, parse_dates=['date'])
    .sort_values('date')
    .set_index('date')
)

df_full = (
    pd.read_csv(url_full, parse_dates=['date'])
    .sort_values('date')
    .set_index('date')
)


In [4]:
df_train.columns

Index(['final.output.concentrate_ag', 'final.output.concentrate_pb',
       'final.output.concentrate_sol', 'final.output.concentrate_au',
       'final.output.recovery', 'final.output.tail_ag', 'final.output.tail_pb',
       'final.output.tail_sol', 'final.output.tail_au',
       'primary_cleaner.input.sulfate', 'primary_cleaner.input.depressant',
       'primary_cleaner.input.feed_size', 'primary_cleaner.input.xanthate',
       'primary_cleaner.output.concentrate_ag',
       'primary_cleaner.output.concentrate_pb',
       'primary_cleaner.output.concentrate_sol',
       'primary_cleaner.output.concentrate_au',
       'primary_cleaner.output.tail_ag', 'primary_cleaner.output.tail_pb',
       'primary_cleaner.output.tail_sol', 'primary_cleaner.output.tail_au',
       'primary_cleaner.state.floatbank8_a_air',
       'primary_cleaner.state.floatbank8_a_level',
       'primary_cleaner.state.floatbank8_b_air',
       'primary_cleaner.state.floatbank8_b_level',
       'primary_cleaner.state

**Validar calculo de recuperación rougher**

Se recalculó la recuperación rougher utilizando las concentraciones de oro en la alimentación, el concentrado rougher y las colas rougher. Tras excluir observaciones con datos faltantes o denominadores no válidos, el EAM entre la recuperación calculada y la registrada fue prácticamente cero. Por lo tanto, se confirma que los valores de rougher.output.recovery fueron calculados correctamente

In [10]:
# Variables de la fórmula
C = df_train['rougher.output.concentrate_au']
F = df_train['rougher.input.feed_au']
T = df_train['rougher.output.tail_au']

# Denominador de la fórmula
denominator = F * (C - T)

# Crear una serie vacía para conservar índice y facilitar el filtrado
calculated_recovery = pd.Series(index=df_train.index, dtype='float64')

# Filas donde es válido calcular la recuperación
valid_denominator = denominator.notna() & (denominator != 0)

# Calcular solo en filas válidas
calculated_recovery.loc[valid_denominator] = (
    C.loc[valid_denominator] * (
        F.loc[valid_denominator] - T.loc[valid_denominator]
    )
    / denominator.loc[valid_denominator]
) * 100

# Crear un DataFrame temporal con valor real y cálculo propio
recovery_comparison = pd.DataFrame({
    'actual_recovery': df_train['rougher.output.recovery'],
    'calculated_recovery': calculated_recovery
})

# Sustituir infinitos por NaN y quedarnos solo con pares válidos
recovery_comparison = (
    recovery_comparison
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Calcular EAM / MAE
mae_recovery = mean_absolute_error(
    recovery_comparison['actual_recovery'],
    recovery_comparison['calculated_recovery']
)

print('Filas evaluadas:', len(recovery_comparison))
print('EAM / MAE:', mae_recovery)


Filas evaluadas: 14287
EAM / MAE: 9.303415616264301e-15


In [11]:
print('NaN en C:', C.isna().sum())
print('NaN en F:', F.isna().sum())
print('NaN en T:', T.isna().sum())
print('Denominadores iguales a cero:', (denominator == 0).sum())
print('Valores infinitos calculados:', np.isinf(calculated_recovery).sum())
print('NaN en recovery real:', df_train['rougher.output.recovery'].isna().sum())

NaN en C: 82
NaN en F: 83
NaN en T: 2249
Denominadores iguales a cero: 63
Valores infinitos calculados: 0
NaN en recovery real: 2573
